In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import re
warnings.filterwarnings('ignore')

In [2]:
def Read_data(filepath):
    df = pd.read_csv(f'{filepath}.csv')
    # df = df[df['fuente'] != 3]
    # df = df[df['calidad'] != 3]
    return df

def rangos_edades(edad):
    if pd.isna(edad):
        return np.nan
    elif edad < 10: 
        return '0-9'
    elif edad < 15:
        return '10-14'
    elif edad < 20:
        return '15-19'
    elif edad < 25:
        return '20-24'
    elif edad < 30:
        return '25-29'
    elif edad < 35:
        return '30-34'
    elif edad < 40:
        return '35-39'
    elif edad < 45:
        return '40-44'
    elif edad >= 45:
        return '45+'
    
def safe_convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return None
    
def Modify_data(df, name_date):

    if 'escolar' in df.columns:
        df['escolar'] = df['escolar'].apply(safe_convert_to_float)
    if 'edad' in df.columns:
        df['edad'] = df['edad'].apply(safe_convert_to_float)
        df['drogaim'] = df['drogaim'].apply(safe_convert_to_float)
    columnas_uc = [
    'tab0uc', 'tab1uc', 'tab2uc',
    'alc0uc',
    'mar0uc', 'mar1uc', 'mar2uc', 'mar3uc',
    'coc0uc', 'coc1uc', 'coc2uc', 'coc3uc',
    'inh0uc', 'inh1uc', 'inh2uc', 'inh3uc', 'inh4uc', 'inh5uc',
    'est0uc', 'est1uc', 'est2uc', 'est3uc', 'est4uc', 'est5uc', 'est6uc',
    'dep0uc', 'dep1uc', 'dep2uc', 'dep3uc', 'dep4uc',
    'alu0uc', 'alu1uc', 'alu2uc', 'alu3uc', 'alu4uc',
    'otr0uc',
    'opi0uc', 'opi1uc', 'opi2uc', 'opi3uc', 'opi4uc', 'opi5uc'
]
    for col in columnas_uc:
        df[col] = df[col].apply(safe_convert_to_float)

    name_date = re.search(r'\\data\\(.*)_1', name_date).group(1)
    year = int(re.search(r'(\d{4})([AB])', name_date).group(1))
    semester = re.search(r'(\d{4})([AB])', name_date).group(2)
    month = 1 if semester == 'A' else 7
    df['Año'] = year
    df['Mes'] = month
    df['Semestre'] = np.where(df['Mes'].between(1, 6), 1, 2)

    cols = [
    'tab0av','tab1av','tab2av','alc0av','mar0av','mar1av','mar3av','mar2av','coc0av','coc1av','coc2av','coc3av','inh0av',
    'inh4av','inh5av','inh1av','inh2av','inh3av','est0av','est5av','est1av','est2av','est3av','est4av','dep2av','dep3av','est6av','alu2av','alu3av',
    'alu0av','alu1av','alu4av','otr0av','dep0av','dep1av','dep4av','opi1av','opi3av','opi0av','opi2av','opi4av','opi5av'
]
    for col in cols:
        df[col] = df[col].astype(int)
        df[col] = df[col].apply(lambda x: x if x in [1, 2] else np.nan)
        df[col] = df[col].replace({2:0})

    dict_estado = pd.read_csv(f'{os.getcwd()}\\data\\CentrosDeCostoEstado.csv')
    dict_estado = dict(zip(dict_estado['CENTRO'], dict_estado['ESTADO']))
    dict3 = {'CIUDAD DE MÉXICO':"CDMX", 'ESTADO DE MÉXICO': "EMEX", 'JALISCO':"JAL", 'SINALOA': 'SIN', 'BAJA CALIFORNIA':"BC", 'CHIHUAHUA': "CHH", 'GUANAJUATO': 'GTO', 'QUINTANA ROO':"QNTROO", 'COAHUILA':  "COAH", 'NUEVO LEÓN':  "NVL", 'MICHOACÁN': "MICH", 'GUERRERO': "GRO", 'COLIMA': "COL", 'BAJA CALIFORNIA SUR':"BCS", 'TAMAULIPAS': "TAM",
        'VERACRUZ': "VRC", 'SONORA': "SON", 'PUEBLA': "PBL", 'DURANGO': "DUR", 'AGUASCALIENTES':  "AGS", 'YUCATÁN':"YCT", 'HIDALGO':"HDO",'ZACATECAS':"ZAC", 'QUERÉTARO': "QRO", 'SAN LUIS POTOSÍ': "SLP" , 'OAXACA': "OAX", 'TABASCO': "TBS", 'CHIAPAS': "CHPS", 'MORELOS':"MOR", 'TLAXCALA': "TLX", 'CAMPECHE':"CAM", 'NAYARIT':"NAY"}
    centros = pd.read_csv(r'C:\Users\franc\Documents\OneDrive\Documentos\GitHub\Recod_Historico_EntrevistaInicial\data\CentrosDeCosto.csv')
    dict_centros = dict(zip(centros['CentroCostoClave'], centros['CentroCostoId']))
    df['CentroCostoId'] = df['cec'].map(dict_centros)
    df['Estado'] = df['cec'].map(dict_estado).map(dict3)
    return df

def min_nonzero(*cols):
    return pd.DataFrame(cols).apply(lambda x: x[x > 0].min() if (x > 0).any() else 0, axis=0)

def Rango_UM(x):
    if x in range(1,4):
        return 1
    elif x == 4:
        return 0
    else: 
        return np.nan

def UltimoMes(df, df2):
    df2['TabacoUM'] = min_nonzero(df['tab0uc'], df['tab1uc'], df['tab2uc'])
    df2['AlcoholUM'] = df['alc0uc']
    df2['MarihuanaUM'] = min_nonzero(df['mar0uc'], df['mar1uc'], df['mar3uc'])
    df2['HachisUM'] = df['mar2uc']
    df2['CocaínaUM'] = min_nonzero(df['coc0uc'], df['coc1uc'])
    df2['CrackUM'] = df['coc2uc']
    df2['Otras Presentaciones (Basuco o pasta base, cocaina negra)UM'] = df['coc3uc']
    df2['Otros (aire comprimido, gasolinas y combustibles)UM'] = min_nonzero(df['inh0uc'], df['inh4uc'], df['inh5uc'])
    df2['Solventes y removedoresUM'] = df['inh1uc']
    df2['PegamentoUM'] = df['inh2uc']
    df2['Esmaltes y pinturasUM'] = df['inh3uc']
    df2['Otros (derivados anfetaminicos)UM'] = min_nonzero(df['est0uc'], df['est5uc'])
    df2['AnfetaminasUM'] = df['est1uc']
    df2['MetanfetaminasUM'] = df['est2uc']
    df2['MDMA(extasis) y metanfetaminas alucinogenas (DMT)UM'] = df['est3uc']
    df2['Otras (sedantes hiptnoticos, GHB)UM'] = min_nonzero(df['est4uc'], df['dep2uc'], df['dep3uc'])
    df2['Otras (PCP, ketamina, excepto metanfetamina)UM'] = min_nonzero(df['est6uc'], df['alu2uc'], df['alu3uc'])
    df2['Plantas alucinogenas y derivadosUM'] = df['alu0uc']
    df2['LSDUM'] = df['alu1uc']
    df2['Otras SustanciasUM'] = min_nonzero(df['alu4uc'], df['otr0uc'])
    df2['BenzodiazepinasUM'] = df['dep0uc']
    df2['RohypnolUM'] = df['dep1uc']
    df2['Opiaceos sinteticos (propoxifeno, nailbufina)UM'] = min_nonzero(df['dep4uc'], df['opi1uc'], df['opi3uc'])
    df2['HeroinaUM'] = df['opi0uc']
    df2['Opio y opiodes (morfina, codeina)UM'] = min_nonzero(df['opi2uc'], df['opi4uc'])
    df2['Con utilidad medica (Prozac, Paxil, Carbamazepina)UM'] = df['opi5uc']
    listcols = [
    'TabacoUM', 'AlcoholUM', 'MarihuanaUM', 'HachisUM', 'CocaínaUM', 'CrackUM', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)UM',
    'Otros (aire comprimido, gasolinas y combustibles)UM', 'Solventes y removedoresUM', 'PegamentoUM', 'Esmaltes y pinturasUM',
    'Otros (derivados anfetaminicos)UM', 'AnfetaminasUM', 'MetanfetaminasUM', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)UM',
    'Otras (sedantes hiptnoticos, GHB)UM', 'Otras (PCP, ketamina, excepto metanfetamina)UM', 'Plantas alucinogenas y derivadosUM', 'LSDUM', 'Otras SustanciasUM',
    'BenzodiazepinasUM', 'RohypnolUM', 'Opiaceos sinteticos (propoxifeno, nailbufina)UM', 'HeroinaUM', 'Opio y opiodes (morfina, codeina)UM', 'Con utilidad medica (Prozac, Paxil, Carbamazepina)UM'
    ]
    for col in listcols:
        df2[col] = df2[col].apply(Rango_UM)
    return df2

def Edad_inicio(df, df2):
    cols_derecha = [
    'tab0e','tab1e','tab2e',
    'alcoe',
    'mar0e','mar1e','mar2e','mar3e',
    'coc0e','coc1e','coc2e','coc3e',
    'inh0e','inh1e','inh2e','inh3e','inh4e','inh5e',
    'est0e','est1e','est2e','est3e','est4e','est5e','est6e',
    'dep0e','dep1e','dep2e','dep3e','dep4e','dep5e',
    'alu0e','alu1e','alu2e','alu3e','alu4e',
    'opi0e','opi1e','opi2e','opi3e','opi4e','opi5e',
    'otr0e','osu3e' ]

    for col in cols_derecha:
        if col not in df.columns:
            df[col] = np.nan

    for col in cols_derecha:
        df[col] = df[col].apply(safe_convert_to_float)
    
    df2['EdadInicioTabaco'] = df[['tab0e','tab1e','tab2e']].min(axis=1)
    df2['EdadInicioAlcohol'] = df['alcoe']
    df2['EdadInicioMarihuana'] = df[['mar0e','mar1e','mar3e']].min(axis=1)
    df2['EdadInicioHachis'] = df['mar2e']
    df2['EdadInicioCocaína'] = df[['coc0e','coc1e']].min(axis=1)
    df2['EdadInicioCrack'] = df['coc2e']
    df2['EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)'] = df['coc3e']
    df2['EdadInicioOtros (aire comprimido, gasolinas y combustibles)'] = df[['inh0e','inh4e','inh5e']].min(axis=1)
    df2['EdadInicioSolventes y removedores'] = df['inh1e']
    df2['EdadInicioPegamento'] = df['inh2e']
    df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
    df2['EdadInicioOtros (derivados anfetaminicos)'] = df[['est0e','est5e']].min(axis=1)
    df2['EdadInicioAnfetaminas'] = df['est1e']
    df2['EdadInicioMetanfetaminas'] = df['est2e']
    df2['EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)'] = df['est3e']
    df2['EdadInicioOtras (sedantes hiptnoticos, GHB)'] = df[['est4e','dep2e','dep3e']].min(axis=1)
    df2['EdadInicioOtras (PCP, ketamina, excepto metanfetamina)'] = df[['est6e','alu2e','alu3e']].min(axis=1)
    df2['EdadInicioPlantas alucinogenas y derivados'] = df['alu0e']
    df2['EdadInicioLSD'] = df['alu1e']
    df2['EdadInicioOtras Sustancias'] = df[['alu4e','otr0e']].min(axis=1)
    df2['EdadInicioBenzodiazepinas'] = df['dep0e']
    df2['EdadInicioRohypnol'] = df['dep1e']
    df2['EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)'] = df[['dep4e','opi1e','opi3e']].min(axis=1)
    df2['EdadInicioHeroina'] = df['opi0e']
    df2['EdadInicioOpio y opiodes (morfina, codeina)'] = df[['opi2e','opi4e']].min(axis=1)
    df2['EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)'] = df['opi5e']
    return df2

def DataEsp(df): 
    list_av_ileg = [
    'Marihuana', 'Hachis', 'Cocaína',
    'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)', 'Otros (aire comprimido, gasolinas y combustibles)', 'Solventes y removedores', 'Pegamento',
    'Esmaltes y pinturas', 'Otros (derivados anfetaminicos)', 'Anfetaminas', 'Otras (sedantes hiptnoticos, GHB)', 'Metanfetaminas',
    'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'Otras Sustancias', 'Otras (PCP, ketamina, excepto metanfetamina)', 'Plantas alucinogenas y derivados', 'LSD',
    'Benzodiazepinas', 'Rohypnol', 'Opio y opiodes (morfina, codeina)', 'Heroina', 'Opiaceos sinteticos (propoxifeno, nailbufina)',
    'Con utilidad medica (Prozac, Paxil, Carbamazepina)'
]
    df['Count_ileg'] = df[list_av_ileg].max(axis=1)
    df['Count_leg'] = (df['Tabaco'] + df['Alcohol']).clip(upper=1)
    df['CasosRiecs'] = np.where(df['Count_ileg'] ==1,1, np.where(df['Count_leg'] ==1,2,0) )
    return df

def Recod_data(df):
    list_aux = ['edocivil', 'escolar', 'ocupa2', 'nivsoc', 'sexo']
    df[list_aux] = (
    df[list_aux]
    .replace(r'^\s*$', np.nan, regex=True)  # ← convierte '' o '   ' en NaN
    .fillna(0)                              # ← rellena NaN con 0
    .astype(int)                            # ← convierte a int
    )
    df2 = pd.DataFrame()
    df2['caso'] = df['caso']
    df2['FolioId'] = df['exp'].astype(str) + '-' + df['cec'].astype(str)
    df2['Edad_Años'] = df['edad']
    df2['Edad'] = df['edad'].apply(rangos_edades)
    df2['Sexo'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer', 9:  np.nan})
    df2['Estado'] = df['Estado']
    df2['CentroCostoId'] = df['CentroCostoId']
    df2 = df2[(df2['CentroCostoId'] > 58) |(df2['CentroCostoId'].isin([48, 49]))]
    df2['Migracion'] = 0
    df2['ComunEstadoCivilId'] = df['edocivil'].map({0: 'Sin Dato', 1: 'Soltero(a)', 2: 'Casado(a)', 3: 'Unión Libre', 4: 'Separado(a)', 5: 'Divorciado(a)', 6: 'Viudo(a)', 9: 'Sin Dato'})
    df2['ComunEscolaridadId'] = df['escolar'].map({0: 'Sin Dato', 10: 'Sin Estudios', 20: 'Sin Estudios', 30: 'Sin Estudios', 31: 'Primaria', 32: 'Sin Estudios', 33: 'Sin Estudios', 40: 'Primaria', 41: 'Secundaria', 42: 'Primaria', 43: 'Primaria', 50: 'Secundaria', 51: 'Preparatoria o Carrera Técnica', 52: 'Secundaria', 53: 'Secundaria', 60: 'Secundaria', 61: 'Preparatoria o Carrera Técnica', 62: 'Secundaria', 63: 'Secundaria', 70: 'Preparatoria o Carrera Técnica', 71: 'Estudios Superiores', 72: 'Preparatoria o Carrera Técnica', 73: 'Preparatoria o Carrera Técnica', 80: 'Estudios Superiores', 81: 'Estudios de Posgrado', 82: 'Estudios Superiores', 83: 'Estudios Superiores'})
    df2['ComunOcupacionId'] = df['ocupa2'].map({0: 'Sin Dato', 1: 'Estudiante', 2: 'Estudiante', 3: 'Con actividad laboral', 4: 'Con actividad laboral', 5: 'Sin ocupación', 6: 'Sin ocupación', 7: 'Hogar', 8: 'Pensionado o jubilado', 9 : 'Sin Dato'})
    df2['ComunEstratoSocialId'] = df['nivsoc'].map({0 : 'Sin Dato', 1: 'Alto', 2: 'Medio Alto', 3: 'Medio Bajo', 4: 'Bajo', 5 : 'Muy Bajo', 9: 'Sin Dato'})
    df2['PerteneceComunidadLGBTTTI'] = 0
    df2['PerteneceComunidadIndigena'] = 0
    df2['PoblacionAfromexicanaAfroamericana'] = 0
    df2['DiscapacidadPerceptual'] = 0
    df2['DrogaImpacto'] = df['drogaim'].replace({0:  np.nan, .1: 'Con utilidad medica (Prozac, Paxil, Carbamazepina)', .2: 'Otros (aire comprimido, gasolinas y combustibles)', .3 : 'Otras Sustancias', .4: 'Otras Sustancias', 1.0:  'Tabaco',1.1:  'Tabaco', 1.2: 'Tabaco', 2.0:  'Alcohol', 3.0:  'Marihuana', 3.1: 'Marihuana', 3.2:  'Hachis', 3.3:'Marihuana', 4.0:  'Cocaína', 4.1:'Cocaína',  4.2:  'Crack', 4.3:  'Otras Presentaciones (Basuco o pasta base, cocaina negra)', 5.0 :  'Otros (aire comprimido, gasolinas y combustibles)', # np.nan es porque lo registran sin Dato
    5.1: 'Solventes y removedores', 5.2: 'Pegamento', 5.3: 'Esmaltes y pinturas', 5.4: 'Otros (aire comprimido, gasolinas y combustibles)', 5.5: 'Otros (aire comprimido, gasolinas y combustibles)' ,6.0: 'Otros (derivados anfetaminicos)', 6.1: 'Anfetaminas', 6.2: 'Metanfetaminas', 6.3: 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
    6.4: 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 6.5: 'Otras (sedantes hiptnoticos, GHB)' , 6.6: 'Otros (derivados anfetaminicos)' , 7.0: 'Otras (PCP, ketamina, excepto metanfetamina)', 7.2: 'Plantas alucinogenas y derivados', 7.1: 'LSD', 7.3: 'Otras (PCP, ketamina, excepto metanfetamina)',
    7.4: 'Otras (PCP, ketamina, excepto metanfetamina)', 8.0: 'Otras Sustancias', 8.1: 'Benzodiazepinas', 8.2: 'Rohypnol', 8.3: 'Otras (sedantes hiptnoticos, GHB)', 8.4: 'Otras (sedantes hiptnoticos, GHB)', 9.0: 'Opiaceos sinteticos (propoxifeno, nailbufina)',
    9.1: 'Heroina' , 9.2: 'Opiaceos sinteticos (propoxifeno, nailbufina)', 9.3: 'Opio y opiodes (morfina, codeina)', 9.4: 'Opiaceos sinteticos (propoxifeno, nailbufina)', 9.5: 'Opio y opiodes (morfina, codeina)', 9.8: 'Con utilidad medica (Prozac, Paxil, Carbamazepina)', 9.9: 'Otras Sustancias',88.0: np.nan ,99: np.nan})
    df2['Año'] = df['Año']
    df2['Mes'] = df['Mes']
    df2['Semestre'] = df2['Mes'].apply(lambda x: 1 if x in range(1, 7) else 2)
    df2['Mes'] = df['Año'].astype(str) + '-' + df['Mes'].astype(str).str.zfill(2)
    df2['Semestre'] = df['Año'].astype(str) + '-' + df['Semestre'].astype(str).str.zfill(2)
    df2['ConsumoDeDrogas'] = df['motivo1']
    df2['ConsumoDeBebidasAlcoholicas'] = df['motivo2']
    df2['ConsumoDeTabaco'] = df['motivo3']
    df2['Ludopatia'] = 0
    if 'motivo4' in df.columns:
        df2['Otro'] = df['motivo4']
    elif 'motivo5a' in df.columns:
        df2['Otro'] = df['motivo5a']
    else:
        df2['Otro'] = 0
    df2['TrastornosMentales'] = 0
    df2['Depresion'] = 0
    df2['Psicosis'] = 0
    df2['Epilepsia'] = 0
    df2['Demencia'] = 0
    df2['Autolesion'] = 0
    df2['Suicidio'] = 0
    df2['Ansiedad'] = 0
    df2['ProblemasSalud'] = df['prob1']
    df2['ProblemasFamiliares'] = df['prob3']
    df2['AccidentesAsociados'] = df['prob2']
    df2['ProblemasEscolares'] = df['prob4']
    df2['ProblemasLaborales'] = df['prob5']
    df2['ProblemasPsicologicos'] = df['prob6']
    df2['ProblemasLegales'] = df['prob7']
    df2['ConductaAntisocial'] = df['prob8']
    df2['ProblemasOtros'] = df['prob9']

    df2['Tabaco'] = df[['tab0av','tab1av','tab2av']].max(axis=1)
    df2['Alcohol'] = df['alc0av'].replace({1:1, 2:0})
    df2['Marihuana'] = df[['mar0av','mar1av','mar3av']].max(axis=1)
    df2['Hachis'] = df['mar2av'].replace({1:1, 2:0})
    df2['Cocaína'] = df[['coc0av','coc1av']].max(axis=1)
    df2['Crack'] = df['coc2av'].replace({1:1, 2:0})
    df2['Otras Presentaciones (Basuco o pasta base, cocaina negra)'] = df['coc3av'].replace({1:1, 2:0})
    df2['Otros (aire comprimido, gasolinas y combustibles)'] = df[['inh0av','inh4av','inh5av']].max(axis=1)
    df2['Solventes y removedores'] = df['inh1av'].replace({1:1, 2:0})
    df2['Pegamento'] = df['inh2av'].replace({1:1, 2:0})
    df2['Esmaltes y pinturas'] = df['inh3av'].replace({1:1, 2:0})
    df2['Otros (derivados anfetaminicos)'] = df[['est0av','est5av']].max(axis=1)
    df2['Anfetaminas'] = df['est1av'].replace({1:1, 2:0})
    df2['Metanfetaminas'] = df['est2av'].replace({1:1, 2:0})
    df2['MDMA(extasis) y metanfetaminas alucinogenas (DMT)'] = df['est3av'].replace({1:1, 2:0})
    df2['Otras (sedantes hiptnoticos, GHB)'] = df[['est4av','dep2av','dep3av']].max(axis=1)
    df2['Otras (PCP, ketamina, excepto metanfetamina)'] = df[['est6av','alu2av','alu3av']].max(axis=1)
    df2['Plantas alucinogenas y derivados'] = df['alu0av'].replace({1:1, 2:0})
    df2['LSD'] = df['alu1av'].replace({1:1, 2:0})
    df2['Otras Sustancias'] = df[['alu4av','otr0av']].max(axis=1)
    df2['Benzodiazepinas'] = df['dep0av'].replace({1:1, 2:0})
    df2['Rohypnol'] = df['dep1av'].replace({1:1, 2:0})
    df2['Opiaceos sinteticos (propoxifeno, nailbufina)']  = df[['dep4av','opi1av','opi3av']].max(axis=1)
    df2['Heroina'] = df['opi0av'].replace({1:1, 2:0})
    df2['Opio y opiodes (morfina, codeina)'] = df[['opi2av','opi4av']].max(axis=1)
    df2['Con utilidad medica (Prozac, Paxil, Carbamazepina)'] = df['opi5av'].replace({1:1, 2:0})
    df2 = UltimoMes(df, df2)
    df2 = Edad_inicio(df, df2)
    df2 = DataEsp(df2)
    return df2

def main ():
    list_files = ['\\data\\2010A_1', '\\data\\2010B_1']
    for file in list_files:
        filepath = os.getcwd() + file
        df = Read_data(filepath)
        df = Modify_data(df, file)
        df = Recod_data(df)
        dfconcat = pd.concat([dfconcat, df]) if 'dfconcat' in locals() else df
    return dfconcat

In [3]:
df = main()
# df.sort_values(by ='Año', inplace=True)
# df = df[df['CasosRiecs'].isin([1,2])]
df = df[df['caso'].isin([1,2])]
df.drop(columns=['CasosRiecs', 'Count_ileg', 'Count_leg'], inplace=True)
# df.drop_duplicates(subset=['FolioId'], keep='last', inplace=True)

In [4]:
df.to_csv(f'{os.getcwd()}\\result\\db_2010.csv', index=False)

In [5]:
df['CentroCostoId'].value_counts()

CentroCostoId
88.0     1292
86.0      733
84.0      724
82.0      690
94.0      660
         ... 
175.0      45
73.0       43
151.0      38
124.0      17
140.0      14
Name: count, Length: 101, dtype: int64